# StockMkt_R — Solution

**Short name:** `StockMkt_R`. Worked answers for Packt Cookbook Ch. 4 (*Modeling Stock Market Data*), extended with alternates, extra practice, a knob-driven simulation, and four-audience notes.

**Disclaimer.** Teaching pipeline only — not a recommendation to buy or sell any security.


## 0. Setup


In [ ]:
library(ggplot2)
library(plyr)
library(reshape2)
library(zoo)
theme_set(theme_minimal())
# dplyr/tidyr are optional; used in the Alternate section
if (requireNamespace("dplyr", quietly = TRUE)) library(dplyr)
if (requireNamespace("tidyr", quietly = TRUE)) library(tidyr)

## 1. Acquire the snapshot


In [ ]:
# Offline snapshot (reproducible). Live Finviz export shown but not required.
finviz <- read.csv("data/finviz.csv", stringsAsFactors = FALSE, check.names = FALSE)
head(finviz[, 1:6])
dim(finviz)
names(finviz)

# Book live pattern (optional):
# url_to_open <- sprintf("http://finviz.com/export.ashx?v=152&c=%s", paste(0:68, collapse = ","))
# finviz_live <- try(read.csv(url(url_to_open)), silent = TRUE)

## 2. Summarize fields


In [ ]:
summary(finviz[, 1:6])
sort(table(finviz$Sector), decreasing = TRUE)

# Vocabulary (also in the cheatsheet)
# Price          last trade
# Volume         shares traded
# P/E            price / earnings per share
# PEG            P/E / expected growth
# Debt/Equity    leverage
# Beta           volatility vs the market (1 = market)
# RSI            0–100 momentum; high ≈ stretched

## 3. Clean numerics


In [ ]:
clean_numeric <- function(s) {
  s <- gsub("%|\\$|,|\\)|\\(", "", s)
  as.numeric(s)
}

id_cols <- 1:6
num_cols <- 7:ncol(finviz)
finviz <- cbind(finviz[, id_cols], as.data.frame(lapply(finviz[, num_cols], clean_numeric)))
# make.names so P/E etc. are legal identifiers
names(finviz) <- make.names(names(finviz))
str(finviz)
summary(finviz$Price)

## 4. Explore the price distribution


In [ ]:
hist(finviz$Price, breaks = 100, main = "Price Distribution (raw)", xlab = "Price")
hist(finviz$Price[finviz$Price < 150], breaks = 100,
     main = "Price Distribution (Price < $150)", xlab = "Price")

sector_avg_prices <- aggregate(Price ~ Sector, data = finviz, FUN = mean)
colnames(sector_avg_prices)[2] <- "Sector_Avg_Price"
ggplot(sector_avg_prices, aes(x = Sector, y = Sector_Avg_Price, fill = Sector)) +
  geom_bar(stat = "identity") +
  theme(axis.text.x = element_text(angle = 35, hjust = 1), legend.position = "none") +
  ggtitle("Sector Avg Price (includes BRK-A)")

## 5. Drill-down and drop BRK-A


In [ ]:
industry_avg_prices <- aggregate(Price ~ Sector + Industry, data = finviz, FUN = mean)
industry_avg_prices <- industry_avg_prices[order(-industry_avg_prices$Price), ]
colnames(industry_avg_prices)[3] <- "Industry_Avg_Price"
industry_chart <- subset(industry_avg_prices, Sector == "Financial")
ggplot(industry_chart, aes(x = Industry, y = Industry_Avg_Price, fill = Industry)) +
  geom_bar(stat = "identity") +
  theme(legend.position = "none", axis.text.x = element_text(angle = 55, hjust = 1)) +
  ggtitle("Financial Industries — Avg Price")

company_chart <- subset(finviz, Industry == "Property & Casualty Insurance")
ggplot(company_chart, aes(x = Company, y = Price, fill = Company)) +
  geom_bar(stat = "identity") +
  theme(legend.position = "none", axis.text.x = element_text(angle = 90, hjust = 1, size = 6)) +
  ggtitle("P&C Insurance — Company Prices")

subset(finviz, Ticker == "BRK-A")[, c("Ticker", "Company", "Price")]

finviz <- subset(finviz, Ticker != "BRK-A")
sector_avg_prices <- aggregate(Price ~ Sector, data = finviz, FUN = mean)
colnames(sector_avg_prices)[2] <- "Sector_Avg_Price"
ggplot(sector_avg_prices, aes(x = Sector, y = Sector_Avg_Price, fill = Sector)) +
  geom_bar(stat = "identity") +
  theme(axis.text.x = element_text(angle = 35, hjust = 1), legend.position = "none") +
  ggtitle("Sector Avg Price (BRK-A removed)")

## 6. Relative valuation averages


In [ ]:
val_vars <- intersect(c("Price", "P.E", "PEG", "P.S", "P.B"), names(finviz))
val_vars  # confirm make.names mapping

sector_avg <- melt(finviz, id = "Sector")
sector_avg <- subset(sector_avg, variable %in% val_vars)
sector_avg <- na.omit(sector_avg)
sector_avg$value <- as.numeric(sector_avg$value)
sector_avg <- dcast(sector_avg, Sector ~ variable, mean)
# rename in a name-safe way
ren <- c(Price = "SAvgPrice", P.E = "SAvgPE", PEG = "SAvgPEG", P.S = "SAvgPS", P.B = "SAvgPB")
for (nm in names(ren)) if (nm %in% names(sector_avg)) {
  names(sector_avg)[names(sector_avg) == nm] <- ren[[nm]]
}
sector_avg

industry_avg <- melt(finviz, id = c("Sector", "Industry"))
industry_avg <- subset(industry_avg, variable %in% val_vars)
industry_avg <- na.omit(industry_avg)
industry_avg$value <- as.numeric(industry_avg$value)
industry_avg <- dcast(industry_avg, Sector + Industry ~ variable, mean)
renI <- c(Price = "IAvgPrice", P.E = "IAvgPE", PEG = "IAvgPEG", P.S = "IAvgPS", P.B = "IAvgPB")
for (nm in names(renI)) if (nm %in% names(industry_avg)) {
  names(industry_avg)[names(industry_avg) == nm] <- renI[[nm]]
}

n_before <- nrow(finviz)
finviz <- merge(finviz, sector_avg, by = "Sector")
finviz <- merge(finviz, industry_avg, by = c("Sector", "Industry"))
c(before = n_before, after = nrow(finviz))
# Rows drop when an industry has no complete valuation fields — acceptable for a screen.

## 7. Flags and RelValIndex


In [ ]:
finviz$SPEUnder <- as.integer(finviz$P.E < finviz$SAvgPE)
finviz$SPEGUnder <- as.integer(finviz$PEG < finviz$SAvgPEG)
finviz$SPSUnder <- as.integer(finviz$P.S < finviz$SAvgPS)
finviz$SPBUnder <- as.integer(finviz$P.B < finviz$SAvgPB)
finviz$SPriceUnder <- as.integer(finviz$Price < finviz$SAvgPrice)
finviz$IPEUnder <- as.integer(finviz$P.E < finviz$IAvgPE)
finviz$IPEGUnder <- as.integer(finviz$PEG < finviz$IAvgPEG)
finviz$IPSUnder <- as.integer(finviz$P.S < finviz$IAvgPS)
finviz$IPBUnder <- as.integer(finviz$P.B < finviz$IAvgPB)
finviz$IPriceUnder <- as.integer(finviz$Price < finviz$IAvgPrice)

flag_cols <- c("SPEUnder","SPEGUnder","SPSUnder","SPBUnder","SPriceUnder",
               "IPEUnder","IPEGUnder","IPSUnder","IPBUnder","IPriceUnder")
finviz$RelValIndex <- rowSums(finviz[flag_cols], na.rm = TRUE)
hist(finviz$RelValIndex, breaks = -0.5:10.5, main = "RelValIndex", xlab = "score")

potentially_undervalued <- subset(finviz, RelValIndex >= 8)
head(potentially_undervalued[order(-potentially_undervalued$RelValIndex),
                             c("Ticker", "Company", "RelValIndex", "Price")], 15)

## 8. Screen + historical prices + MAs


In [ ]:
# Column names after make.names on this snapshot
names(finviz)

target_stocks <- subset(
  finviz,
  Price > 20 & Price < 100 &
    Volume > 10000 &
    Country == "USA" &
    EPS..ttm. > 0 &
    EPS.growth.next.year > 0 &
    EPS.growth.next.5.years > 0 &
    Total.Debt.Equity < 1 &
    Beta < 1.5 &
    Institutional.Ownership < 30 &
    RelValIndex >= 8
)
target_stocks[, c("Ticker", "Company", "RelValIndex", "Price", "Sector")]

hist_px <- read.csv("data/historical_prices.csv", stringsAsFactors = FALSE)
hist_px$Date <- as.Date(hist_px$Date)

# One-name MA chart (DOW if present, else first target)
sym <- if ("DOW" %in% hist_px$Symbol) "DOW" else unique(hist_px$Symbol)[1]
one <- subset(hist_px, Symbol == sym)
one <- one[order(one$Date), ]
one$MovAvg50 <- NA
one$MovAvg200 <- NA
if (nrow(one) >= 50) {
  one$MovAvg50[50:nrow(one)] <- rollmean(one$AdjClose, 50, align = "right")
}
if (nrow(one) >= 200) {
  one$MovAvg200[200:nrow(one)] <- rollmean(one$AdjClose, 200, align = "right")
}
pc <- melt(one[, c("Date", "AdjClose", "MovAvg50", "MovAvg200")], id = "Date")
ggplot(pc, aes(Date, value, color = variable)) +
  geom_line() +
  ggtitle(paste(sym, "AdjClose + 50/200-day MAs")) +
  ylab("Price")

ggplot(hist_px, aes(Date, AdjClose, color = Symbol)) +
  geom_line() +
  ggtitle("Target names — daily AdjClose")

price_summaries <- ddply(hist_px, "Symbol", summarise,
                         open = Open[1],
                         high = max(High, na.rm = TRUE),
                         low = min(Low, na.rm = TRUE),
                         close = AdjClose[length(AdjClose)])
summary_long <- melt(price_summaries, id = "Symbol")
ggplot(summary_long, aes(variable, value, fill = Symbol)) +
  geom_bar(stat = "identity", position = "dodge") +
  facet_wrap(~ Symbol, scales = "free_y") +
  ggtitle("Open / High / Low / Close (window)") +
  theme(legend.position = "none")

## Alternate code


In [ ]:
# A. dplyr sector means (same as aggregate)
if (requireNamespace("dplyr", quietly = TRUE)) {
  sector_dplyr <- dplyr::summarise(dplyr::group_by(finviz, Sector),
                                   Sector_Avg_Price = mean(Price, na.rm = TRUE))
  print(head(sector_dplyr))
}

# B. tidyr pivot instead of melt/dcast for sector PE mean
if (requireNamespace("tidyr", quietly = TRUE) && requireNamespace("dplyr", quietly = TRUE)) {
  sector_pe <- finviz |>
    dplyr::select(Sector, P.E) |>
    tidyr::drop_na() |>
    dplyr::group_by(Sector) |>
    dplyr::summarise(SAvgPE = mean(P.E), .groups = "drop")
  print(sector_pe)
}

# C. plyr::ddply — the book's own suggested alternate for aggregate
sector_ddply <- ddply(finviz, "Sector", summarise, Price = mean(Price, na.rm = TRUE))
print(sector_ddply)

# D. base filter MA (no zoo) for the same symbol
k <- 50
w <- rep(1 / k, k)
ma_base <- as.numeric(stats::filter(one$AdjClose, w, sides = 1))
# last value should be close to zoo rollmean
tail(na.omit(ma_base), 1)
tail(na.omit(one$MovAvg50), 1)

## More practice


In [ ]:
# 1. Median-based index
med_sector <- aggregate(cbind(Price, P.E, PEG, P.S, P.B) ~ Sector, data = finviz, FUN = median)
names(med_sector)[-1] <- paste0("MS_", names(med_sector)[-1])
tmp <- merge(finviz, med_sector, by = "Sector")
tmp$MedIdx <- (tmp$P.E < tmp$MS_P.E) + (tmp$PEG < tmp$MS_PEG) +
  (tmp$P.S < tmp$MS_P.S) + (tmp$P.B < tmp$MS_P.B) + (tmp$Price < tmp$MS_Price)
c(mean_ge8 = sum(finviz$RelValIndex >= 8, na.rm = TRUE),
  median_ge5 = sum(tmp$MedIdx >= 4, na.rm = TRUE))

# 2. Quality extras (index 0–12 idea)
finviz$QBeta <- as.integer(finviz$Beta < 1)
finviz$QRSI <- as.integer(finviz$RSI < 40)
finviz$RelValPlus <- finviz$RelValIndex + finviz$QBeta + finviz$QRSI
table(finviz$RelValPlus)

# 3. Return correlations among bundled histories
wide <- dcast(hist_px, Date ~ Symbol, value.var = "AdjClose")
rets <- as.data.frame(lapply(wide[-1], function(x) diff(log(x))))
print(round(cor(rets, use = "pairwise.complete.obs"), 2))

# 4. PE boxes Tech vs Utilities
pe_box <- subset(finviz, Sector %in% c("Technology", "Utilities") & P.E < 200)
ggplot(pe_box, aes(Sector, P.E, fill = Sector)) +
  geom_boxplot(outlier.alpha = 0.3) +
  ggtitle("P/E: Technology vs Utilities (P/E < 200)")

## Simulation / what-if


In [ ]:
idx_cut <- 8
price_lo <- 20
price_hi <- 100
max_de <- 1
max_beta <- 1.5
max_inst <- 30
usa_only <- TRUE
noise_sd <- 0
set.seed(42)

sim <- finviz
if (noise_sd > 0) sim$Price <- sim$Price + rnorm(nrow(sim), 0, noise_sd)

keep_usa <- if (usa_only) sim$Country == "USA" else TRUE
passers <- subset(
  sim,
  keep_usa &
    Price > price_lo & Price < price_hi &
    Volume > 10000 &
    EPS..ttm. > 0 &
    EPS.growth.next.year > 0 &
    EPS.growth.next.5.years > 0 &
    Total.Debt.Equity < max_de &
    Beta < max_beta &
    Institutional.Ownership < max_inst &
    RelValIndex >= idx_cut
)
n_pass <- nrow(passers)
n_pass
if (n_pass) {
  print(median(passers$Price, na.rm = TRUE))
  print(sort(table(passers$Sector), decreasing = TRUE))
}

# Threshold sweep
cuts <- 4:10
n_by_cut <- sapply(cuts, function(k) sum(sim$RelValIndex >= k, na.rm = TRUE))
sweep <- data.frame(idx_cut = cuts, n = n_by_cut)
ggplot(sweep, aes(idx_cut, n)) +
  geom_line() + geom_point() +
  ggtitle("Names with RelValIndex ≥ cutoff") +
  xlab("cutoff") + ylab("count")

## Audience rewrite

**Expert / quant.** RelValIndex is an unweighted count of ten “below cross-sectional mean” events on Price, P/E, PEG, P/S and P/B at sector and industry grain. Means are not robust — BRK-A is the worked example — and the industry merge drops names with incomplete comps, a form of selection. Flags treat a 1 bp gap the same as a 40% gap. There is no look-ahead in the snapshot itself, but any *historical* MA rule used as a timing overlay is in-sample. Use medians or winsorized means, and do not read RelValIndex ≥ 8 as expected alpha.

**Technician / screener operator.** Pipeline: `read.csv("data/finviz.csv")` → `clean_numeric` on columns 7+ → `make.names` → drop `Ticker == "BRK-A"` → melt/dcast (or dplyr) sector and industry means of Price/PE/PEG/PS/PB → ten `< avg` flags → `rowSums` → subset USA, Price 20–100, Volume > 10k, EPS and both growth fields > 0, D/E < 1, Beta < 1.5, InstOwn < 30, index ≥ 8. Histories live in `data/historical_prices.csv`. Re-run after replacing the snapshot; if column count changes, stop using hard indexes.

**Executive / CIO.** One Class-A outlier made Financials look “expensive” on a mean-price bar. After removing it, sector average prices sit in a narrow band. A simple neighbor-comparison score plus a conservative quality screen leaves a short US list — a conversation starter, not a portfolio. Moving-average charts show path and volatility; they do not forecast next quarter’s earnings.

**Nonspecialist.** Imagine comparing a shop’s sticker price to other shops on the same street, not to a secret “true” value. We also threw out one extreme price that was warping the street average, kept only US names in a mid price range with manageable debt, and then looked at how those prices moved over a few years. That is a screening story, not a promise.


## Key takeaways

- Reproducible *code* beats a point-and-click Finviz export after the first pass.
- `%` `$` `,` turn numbers into strings — `gsub` + `as.numeric` is the whole game.
- Means without an outlier check lie; BRK-A is the canonical demo.
- Relative value is a *comparison*, not an intrinsic model. Ten binary flags are a teaching index, not a factor model.
- Screens exist to shrink the list. Historical MAs then describe path and volatility of whatever survived.
- Live vendor URLs rot. Bundle a snapshot.
